# Weekly Monitoring Scorecard

## Business question

What changed this week in complaint volume, risk, and topic mix, and what should be escalated?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

scores_path = Path("reports/sample/complaint_scores.csv")
terms_path = Path("reports/sample/topic_terms.csv")
if not scores_path.exists() or not terms_path.exists():
    raise FileNotFoundError(
        "Run `cx-complaints analyze --output-dir reports/sample` first."
    )

scores = pd.read_csv(scores_path)
topic_terms = pd.read_csv(terms_path)
scores["date_received"] = pd.to_datetime(scores["date_received"], errors="coerce")
scores = scores.dropna(subset=["date_received"]).copy()
scores["week"] = scores["date_received"].dt.to_period("W").astype(str)

plt.style.use("ggplot")

## 1. Weekly KPI table

In [ ]:
weekly = (
    scores.groupby("week")
    .agg(
        complaints=("complaint_id", "size"),
        high_risk=("risk_level", lambda s: (s == "high").sum()),
        medium_or_high=("risk_level", lambda s: s.isin(["medium", "high"]).sum()),
        avg_risk_score=("risk_score", "mean"),
        untimely=("timely", lambda s: (s.fillna("").str.lower() != "yes").sum()),
    )
    .reset_index()
    .sort_values("week")
)

weekly["high_risk_share_pct"] = (weekly["high_risk"] / weekly["complaints"] * 100).round(1)
weekly["medium_or_high_share_pct"] = (
    weekly["medium_or_high"] / weekly["complaints"] * 100
).round(1)
weekly["untimely_share_pct"] = (weekly["untimely"] / weekly["complaints"] * 100).round(1)
weekly["avg_risk_score"] = weekly["avg_risk_score"].round(2)
weekly.tail(12)

In [ ]:
if len(weekly) >= 2:
    current = weekly.iloc[-1]
    previous = weekly.iloc[-2]
    wow = pd.DataFrame(
        {
            "metric": ["complaints", "high_risk_share_pct", "untimely_share_pct", "avg_risk_score"],
            "current": [
                current["complaints"],
                current["high_risk_share_pct"],
                current["untimely_share_pct"],
                current["avg_risk_score"],
            ],
            "previous": [
                previous["complaints"],
                previous["high_risk_share_pct"],
                previous["untimely_share_pct"],
                previous["avg_risk_score"],
            ],
        }
    )
    wow["delta"] = (wow["current"] - wow["previous"]).round(2)
    print(wow)
else:
    print("Need at least 2 weeks to compute week-over-week deltas.")

## 2. Weekly trend charts

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(weekly["week"], weekly["complaints"], marker="o", label="Complaints")
axes[0].plot(weekly["week"], weekly["high_risk"], marker="s", label="High-risk complaints")
axes[0].set_title("Weekly Complaint and High-Risk Volume")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].plot(weekly["week"], weekly["high_risk_share_pct"], marker="o", label="High-risk share (%)")
axes[1].plot(weekly["week"], weekly["untimely_share_pct"], marker="s", label="Untimely share (%)")
axes[1].set_title("Weekly Risk and Response-Quality Shares")
axes[1].set_xlabel("Week")
axes[1].set_ylabel("Share (%)")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()

## 3. Topic movement this week

In [ ]:
topic_labels = (
    topic_terms.sort_values(["topic_id", "rank"])
    .groupby("topic_id")["term"]
    .apply(lambda s: ", ".join(s.head(4).astype(str)))
    .rename("topic_label")
    .reset_index()
)

topic_weekly = (
    scores.groupby(["week", "topic_id"]).size().rename("complaints").reset_index()
)
topic_weekly = topic_weekly.merge(topic_labels, on="topic_id", how="left")

if scores["week"].nunique() >= 2:
    latest_week = sorted(scores["week"].unique())[-1]
    prev_week = sorted(scores["week"].unique())[-2]

    latest_topic = topic_weekly[topic_weekly["week"] == latest_week][["topic_id", "complaints"]]
    latest_topic = latest_topic.rename(columns={"complaints": "latest_complaints"})

    prev_topic = topic_weekly[topic_weekly["week"] == prev_week][["topic_id", "complaints"]]
    prev_topic = prev_topic.rename(columns={"complaints": "prev_complaints"})

    topic_move = topic_labels.merge(latest_topic, on="topic_id", how="left")\
        .merge(prev_topic, on="topic_id", how="left")
    topic_move[["latest_complaints", "prev_complaints"]] = (
        topic_move[["latest_complaints", "prev_complaints"]].fillna(0)
    )
    topic_move["delta_complaints"] = topic_move["latest_complaints"] - topic_move["prev_complaints"]
    topic_move.sort_values("delta_complaints", ascending=False).head(12)
else:
    print("Need at least 2 weeks to compare topic movement.")

## 4. Escalation checklist

Escalate when one or more occur:

1. complaint volume rises week-over-week by a defined threshold;
2. high-risk share or untimely share increases meaningfully;
3. one topic shows a sustained week-over-week rise.

In [ ]:
# Optional export
output_dir = Path("reports/notebook")
output_dir.mkdir(parents=True, exist_ok=True)

weekly.to_csv(output_dir / "notebook_weekly_scorecard.csv", index=False)
topic_weekly.to_csv(output_dir / "notebook_topic_weekly_counts.csv", index=False)

output_dir